# Train Vidgin — real videos first, smoke test second

This notebook has two clearly separated paths:

1. **REAL-VIDEO TRAINING (the primary path, Part B)** — trains Vidgin on
   your prepared "City and people" clips: dataset health check, train/test
   split, a caption tokenizer built from your own captions, VAE training,
   a measured latent scale, DiT training, evaluation, and an MP4 sample.
2. **SMOKE TEST (Part A)** — a tiny synthetic Moving MNIST run that only
   proves the plumbing works. It is NOT real training and its results
   must never be reported as real-video results.

Run the Setup cells, then Part A if you want the quick plumbing check,
then Part B for the real run.

In [ ]:
# --- Setup: check the GPU ---
import torch
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - CPU only (fine for the smoke test, too slow for real training)")

In [ ]:
# --- Get the code (clones your repo; safe to re-run) ---
import os, subprocess
if not os.path.isdir("Vidgin"):
    subprocess.run(["git", "clone", "https://github.com/shahrozkalash/Vidgin"], check=True)
os.chdir("Vidgin")
print("working dir:", os.getcwd())

In [ ]:
# --- Optional sanity check: run the test suite (a few minutes) ---
# Proves imports, datasets, trainers, tokenizer, and checkpointing work
# in THIS environment before any long run starts.
# !python -m pytest tests/ -q

## Part A — SMOKE TEST ONLY (synthetic Moving MNIST)

A few hundred steps on fake data generated in code. Purpose: prove the
trainers run end-to-end in this environment. **Nothing here is real
training** — the trainers print a loud warning whenever they run on
synthetic data, and `--require-real` (used in Part B) turns that
fallback into an error.

In [ ]:
# --- Smoke: tiny VAE + DiT, a few hundred steps each ---
!python -m vidgen.train_vae --device cuda --preset tiny --steps 200 --ckpt-dir checkpoints/smoke-vae --log smoke_vae.jsonl
!python -m vidgen.train --device cuda --preset tiny --steps 500 --vae-ckpt checkpoints/smoke-vae/latest.pt --ckpt-dir checkpoints/smoke-dit --log smoke_dit.jsonl
print("smoke run finished - plumbing only, NOT real training")

## Part B — REAL-VIDEO TRAINING (primary path)

Set the variables in the next cell. `DATA_DIR` is a prepared dataset
directory (manifest.jsonl + .npy clips, built by
`scripts/prepare_clips.py`, e.g. on Drive at
`vidgin-data/small-clips` after packing). Every training command below
passes `--require-real`, so a missing dataset stops the run loudly
instead of silently training on synthetic data.

In [ ]:
# --- Real-run settings (edit DATA_DIR / PRESET for your run) ---
DATA_DIR = "/content/drive/MyDrive/vidgin-data/small-clips/packed"  # prepared + packed dataset
PRESET = "small"          # 256x256, 33 frames, depth-conditioned
FRAMES, SIZE = 33, 256    # must match how the clips were prepared
VAE_INIT = ""             # optional: warm-start VAE weights (e.g. the Round 1 release VAE); "" = from scratch
import os
print("dataset exists:", os.path.isdir(DATA_DIR))

In [ ]:
# --- Step B1: check the dataset, then make the train/val/test split ---
!python scripts/vidgin_helper.py check {DATA_DIR} --frames {FRAMES} --height {SIZE} --width {SIZE} --require-depth
!python scripts/vidgin_helper.py split {DATA_DIR} --ratios 0.8,0.1,0.1 --seed 1234

In [ ]:
# --- Step B2: build the caption tokenizer from YOUR captions ---
!python scripts/build_tokenizer.py --data-dir {DATA_DIR} --out tokenizer.json

In [ ]:
# --- Step B3: train the VAE on real clips (split: train) ---
# Warm start is optional; --require-real makes a missing dataset an error.
init_flag = f"--init-weights {VAE_INIT}" if VAE_INIT else ""
!python -m vidgen.train_vae --device cuda --preset {PRESET} --data-dir {DATA_DIR} --data-split train --require-real {init_flag} --steps 4000 --batch-size 2 --ckpt-dir checkpoints/real-vae --log real_vae.jsonl

In [ ]:
# --- Step B4: MEASURE the VAE before DiT training ---
# Reports the latent std, the recommended --latent-scale, reconstruction
# PSNR, and preview images. If PSNR is very low, fix the VAE first -
# the DiT can only learn from the latents this VAE produces.
!python scripts/measure_latents.py --vae-ckpt checkpoints/real-vae/latest.pt --preset {PRESET} --data-dir {DATA_DIR} --max-clips 200 --out latent_report.json --preview-dir latent_previews
import json
LATENT_SCALE = json.load(open("latent_report.json"))["recommended_latent_scale"]
print("measured latent scale:", LATENT_SCALE)

In [ ]:
# --- Step B5: train the DiT on real clips ---
# Uses the measured latent scale, the caption tokenizer, and depth
# conditioning. Checkpoints land in checkpoints/real-dit.
!python -m vidgen.train --device cuda --preset {PRESET} --data-dir {DATA_DIR} --data-split train --require-real --depth --grad-checkpoint --batch-size 2 --vae-ckpt checkpoints/real-vae/latest.pt --tokenizer-vocab tokenizer.json --latent-scale {LATENT_SCALE} --steps 15000 --ckpt-dir checkpoints/real-dit --log real_dit.jsonl

In [ ]:
# --- Step B6: progress - first vs last logged loss ---
import json
for name in ("real_vae.jsonl", "real_dit.jsonl"):
    try:
        rows = [json.loads(l) for l in open(name) if l.strip()]
        print(name, "steps:", len(rows), "first loss:", round(rows[0]["loss"], 4), "last loss:", round(rows[-1]["loss"], 4))
    except FileNotFoundError:
        print(name, "not found (stage not run yet)")

In [ ]:
# --- Step B7: evaluate the trained checkpoint ---
# Fixed prompts + seeds, saved MP4s, non-learned quality metrics, and a
# report. Add --baseline <older dit.pt> to compare against a previous run.
!python scripts/evaluate_checkpoint.py --dit-ckpt checkpoints/real-dit/latest.pt --vae-ckpt checkpoints/real-vae/latest.pt --train-dir {DATA_DIR} --steps 12 --out-dir eval_out

In [ ]:
# --- Step B8: generate a video (MP4 written next to the frames) ---
PROMPT = "people walking across a busy city street with traffic"
!python -m vidgen.sample --device cuda --ckpt checkpoints/real-dit/latest.pt --vae-ckpt checkpoints/real-vae/latest.pt --prompt "{PROMPT}" --outdir sample_out
print("watch: sample_out/clip.mp4")

## Session ended mid-training? Resume here

Re-run the **Setup** cells first, then the resume command for your
stage. `--steps` after `--resume` counts ADDITIONAL steps. The
checkpoint restores the tokenizer, latent scale, LR schedule, and
dataset split by itself — do not change those flags on resume.

In [ ]:
# --- Resume commands (uncomment the stage you need) ---
# VAE:  !python -m vidgen.train_vae --device cuda --resume checkpoints/real-vae/latest.pt --data-dir {DATA_DIR} --data-split train --require-real --steps 2000 --ckpt-dir checkpoints/real-vae --log real_vae.jsonl
# DiT:  !python -m vidgen.train --device cuda --resume checkpoints/real-dit/latest.pt --data-dir {DATA_DIR} --data-split train --require-real --steps 5000 --ckpt-dir checkpoints/real-dit --log real_dit.jsonl

In [ ]:
# --- Save your trained models: zip the checkpoints ---
# Colab/Kaggle: download the zip; the checkpoints also belong in Drive.
!zip -r vidgin_checkpoints.zip checkpoints tokenizer.json latent_report.json
print("zip ready: vidgin_checkpoints.zip")

## What you have now

A Vidgin model **you trained on real videos**: `checkpoints/real-dit`
(+ its VAE in `checkpoints/real-vae`), an evaluation report in
`eval_out/` with MP4 samples, the measured latent report, and the
caption tokenizer — all of it Vidgin's own, trained from its own code
and data. The smoke-test checkpoints under `checkpoints/smoke-*` are
plumbing proof only.